In [83]:
import numpy as np
from qiskit.circuit.library import zz_feature_map
from qiskit_machine_learning.kernels import FidelityQuantumKernel
from sklearn.datasets import load_breast_cancer
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, recall_score 
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.svm import SVC
import time

# Setup dataset
dataset = load_breast_cancer()
X = dataset.data
y = dataset.target

# Split data into training and testing
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

sample_size = 20
features_size = 4

# Select features using training data only
selector = SelectKBest(score_func=f_classif, k=features_size)

X_train_selected = selector.fit_transform(X_train, y_train)
X_test_selected = selector.transform(X_test)

# Display selected features
selected_features = dataset.feature_names[selector.get_support()]
print("Selected features:")
for feature in selected_features:
    print("-", feature)

# Scale from 0 to pi
scaler = MinMaxScaler(feature_range=(0, np.pi))
X_train_scaled = scaler.fit_transform(X_train_selected)
X_test_scaled = scaler.transform(X_test_selected)

X_train_subset = X_train_scaled[:sample_size]
y_train_subset = y_train[:sample_size]

# Keep the full test set for evaluation
X_test_subset = X_test_scaled
y_test_subset = y_test

# Encode features into qubits
feature_map = zz_feature_map(
    feature_dimension = features_size,
    reps = 1,
    entanglement = "linear"
)

# Calculate quantum kernel
quantum_kernel = FidelityQuantumKernel(
    feature_map=feature_map
)

Selected features:
- mean concave points
- worst radius
- worst perimeter
- worst concave points


In [84]:
start_kernel_train = time.time()

K_train = quantum_kernel.evaluate(
    x_vec=X_train_subset[:sample_size]
)

print("Training kernel time (seconds):", round(time.time() - start_kernel_train, 2))
print(K_train.shape)

Training kernel time (seconds): 0.41
(20, 20)


In [85]:
start_kernel_test = time.time()

K_test = quantum_kernel.evaluate(
    x_vec=X_test_subset,
    y_vec=X_train_subset[:sample_size]
)

print("Test kernel time (seconds):", round(time.time() - start_kernel_test, 2))
print(K_test.shape)

Test kernel time (seconds): 5.75
(114, 20)


In [82]:
qsvc = SVC(kernel="precomputed")
qsvc.fit(K_train, y_train[:sample_size])

y_pred = qsvc.predict(K_test)

print("Accuracy:", accuracy_score(y_test, y_pred))

print("Classification report:")
print(classification_report(
    y_test,
    y_pred,
    target_names=dataset.target_names,
    zero_division=0
))

print("Malignant recall:", recall_score(
    y_test,
    y_pred,
    pos_label=0,
    zero_division=0
))

Accuracy: 0.7894736842105263
Classification report:
              precision    recall  f1-score   support

   malignant       0.91      0.48      0.62        42
      benign       0.76      0.97      0.85        72

    accuracy                           0.79       114
   macro avg       0.83      0.72      0.74       114
weighted avg       0.82      0.79      0.77       114

Malignant recall: 0.47619047619047616
